[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_13/NB_bishop_ch13_fraud_graph.ipynb)

# Chapter 13: Transaction Fraud Detection with Graph Features

**Bishop & Bishop (2024), Chapter 13**

We build a synthetic transaction graph (accounts = nodes, transactions = edges),
extract graph-structural features (degree centrality, clustering coefficient,
PageRank), and train a Logistic Regression classifier to detect fraudulent accounts.

**Course:** Neural Networks and Deep Learning with Business Applications

**Author:** Daniel Traian PELE -- Bucharest University of Economic Studies

In [ ]:
# Run in Google Colab to install dependencies
!pip install -q networkx


In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import networkx as nx
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_curve, auc, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay
)

# Course colors
BLUE    = '#1f4e79'
CRIMSON = '#c00000'
GREEN   = '#2e7d32'

matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor']   = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['figure.figsize'] = (10, 5)

np.random.seed(42)

def save_fig(fig, name, chart_dir='../../charts'):
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            ncol = min(len(legend.get_texts()), 4)
            legend._ncols = ncol
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

print('Chapter 13: Fraud Detection with Graph Features -- Ready!')

## 1. Generate a Synthetic Transaction Graph

- **500 nodes** (accounts)
- **~2 000 edges** (transactions)
- **5 % fraud nodes** that form a denser sub-community with distinctive
  connectivity patterns (higher internal density, unusual degree distribution).

In [ ]:
np.random.seed(42)

N_NODES = 500
FRAUD_RATE = 0.05
n_fraud = int(N_NODES * FRAUD_RATE)  # 25 fraud nodes
n_legit = N_NODES - n_fraud

# Assign labels: last n_fraud nodes are fraudulent
labels = np.zeros(N_NODES, dtype=int)
labels[-n_fraud:] = 1
fraud_nodes = set(np.where(labels == 1)[0])
legit_nodes = set(np.where(labels == 0)[0])

G = nx.Graph()
G.add_nodes_from(range(N_NODES))

edges_added = 0

# --- Legitimate edges: sparse random connections ---
target_legit_edges = 1700
legit_list = list(legit_nodes)
while edges_added < target_legit_edges:
    u = np.random.choice(legit_list)
    v = np.random.choice(N_NODES)  # can connect to anyone
    if u != v and not G.has_edge(u, v):
        G.add_edge(u, v, weight=np.random.uniform(100, 10000))
        edges_added += 1

# --- Fraud ring: dense internal connections + some external ---
fraud_list = list(fraud_nodes)
# Dense internal ring
for i in range(n_fraud):
    for j in range(i + 1, n_fraud):
        if np.random.rand() < 0.45:  # 45% internal density
            u, v = fraud_list[i], fraud_list[j]
            if not G.has_edge(u, v):
                G.add_edge(u, v, weight=np.random.uniform(50, 5000))

# Some connections from fraud to legit (camouflage)
for f_node in fraud_list:
    n_ext = np.random.randint(1, 5)
    for _ in range(n_ext):
        v = np.random.choice(legit_list)
        if not G.has_edge(f_node, v):
            G.add_edge(f_node, v, weight=np.random.uniform(100, 10000))

print(f'Transaction graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print(f'Fraud nodes: {n_fraud} ({FRAUD_RATE*100:.0f}%)')
print(f'Legitimate nodes: {n_legit}')

## 2. Compute Graph Features

For each node we compute structural features that a GNN would learn implicitly:

| Feature | Intuition |
|---|---|
| **Degree centrality** | How many connections an account has |
| **Clustering coefficient** | How densely an account's neighbours are interconnected |
| **PageRank** | Overall importance / influence in the network |
| **Average neighbour degree** | Whether an account connects to hubs or periphery |

In [ ]:
# Compute graph features
degree_cent  = nx.degree_centrality(G)
clustering   = nx.clustering(G)
pagerank     = nx.pagerank(G, alpha=0.85)
avg_nb_deg   = nx.average_neighbor_degree(G)

df = pd.DataFrame({
    'node':          range(N_NODES),
    'degree_cent':   [degree_cent[i]  for i in range(N_NODES)],
    'clustering':    [clustering[i]   for i in range(N_NODES)],
    'pagerank':      [pagerank[i]     for i in range(N_NODES)],
    'avg_nb_degree': [avg_nb_deg[i]   for i in range(N_NODES)],
    'fraud':         labels
})

print('Feature statistics by class:')
print(df.groupby('fraud')[['degree_cent', 'clustering', 'pagerank', 'avg_nb_degree']].mean().round(5))

In [ ]:
# Feature distributions
feat_cols = ['degree_cent', 'clustering', 'pagerank', 'avg_nb_degree']
feat_labels = ['Degree Centrality', 'Clustering Coeff.', 'PageRank', 'Avg. Neighbour Degree']

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col, lab in zip(axes, feat_cols, feat_labels):
    ax.hist(df.loc[df.fraud == 0, col], bins=25, alpha=0.7, color=BLUE,
            label='Legitimate', density=True)
    ax.hist(df.loc[df.fraud == 1, col], bins=15, alpha=0.7, color=CRIMSON,
            label='Fraud', density=True)
    ax.set_xlabel(lab)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

fig.suptitle('Graph Feature Distributions: Fraud vs Legitimate', fontsize=13, y=1.02)
fig.tight_layout()
save_fig(fig, 'bishop_ch13_fraud_features')
plt.show()

## 3. Train Logistic Regression on Graph Features

In [ ]:
X = df[feat_cols].values
y = df['fraud'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

clf = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

print('Classification Report (test set):')
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))

# Feature importance (coefficients)
print('Logistic Regression Coefficients:')
for name, coef in zip(feat_cols, clf.coef_[0]):
    print(f'  {name:20s}: {coef:+.4f}')

## 4. ROC Curve and Confusion Matrix

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
cm = confusion_matrix(y_test, y_pred)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# ROC
ax1.plot(fpr, tpr, color=CRIMSON, linewidth=2, label=f'AUC = {roc_auc:.3f}')
ax1.plot([0, 1], [0, 1], '--', color='gray', linewidth=1)
ax1.set_xlabel('False Positive Rate', fontsize=11)
ax1.set_ylabel('True Positive Rate', fontsize=11)
ax1.set_title('ROC Curve -- Fraud Detection', fontsize=12)
ax1.legend(fontsize=11)

# Confusion matrix
disp = ConfusionMatrixDisplay(cm, display_labels=['Legitimate', 'Fraud'])
disp.plot(ax=ax2, cmap='Blues', colorbar=False)
ax2.set_title('Confusion Matrix', fontsize=12)

fig.tight_layout()
save_fig(fig, 'bishop_ch13_fraud_roc_cm')
plt.show()

## 5. Visualize the Fraud Sub-Graph

We extract the sub-graph induced by all fraud nodes plus their immediate
neighbours and highlight the fraud ring.

In [ ]:
# Sub-graph: fraud nodes + their 1-hop neighbours
fraud_and_neighbours = set(fraud_nodes)
for f in fraud_nodes:
    fraud_and_neighbours.update(G.neighbors(f))

G_sub = G.subgraph(fraud_and_neighbours).copy()

node_colors_sub = []
node_sizes_sub  = []
for n in G_sub.nodes():
    if n in fraud_nodes:
        node_colors_sub.append(CRIMSON)
        node_sizes_sub.append(180)
    else:
        node_colors_sub.append(BLUE)
        node_sizes_sub.append(60)

# Edge colors: red if both endpoints are fraud
edge_colors_sub = []
for u, v in G_sub.edges():
    if u in fraud_nodes and v in fraud_nodes:
        edge_colors_sub.append(CRIMSON)
    else:
        edge_colors_sub.append('#cccccc')

pos_sub = nx.spring_layout(G_sub, seed=42, k=0.3)

fig, ax = plt.subplots(figsize=(10, 8))
nx.draw(G_sub, pos_sub, ax=ax,
        node_color=node_colors_sub, node_size=node_sizes_sub,
        edge_color=edge_colors_sub, width=0.6,
        with_labels=False, alpha=0.85)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=CRIMSON, markersize=10, label='Fraud account'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=BLUE,    markersize=7,  label='Legitimate neighbour'),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=10)
ax.set_title('Fraud Sub-Graph: Fraud Ring + 1-Hop Neighbours', fontsize=13)

fig.tight_layout()
save_fig(fig, 'bishop_ch13_fraud_subgraph')
plt.show()

print(f'Sub-graph: {G_sub.number_of_nodes()} nodes, {G_sub.number_of_edges()} edges')

## Key Takeaways

1. **Graph structure is informative.** Fraud accounts exhibit higher clustering
   coefficients and different centrality profiles than legitimate accounts,
   because they form dense internal rings.

2. **Hand-crafted graph features** (degree centrality, clustering, PageRank)
   already provide strong signal. A GNN learns analogous features
   *automatically* via message passing.

3. **Class imbalance** is a key challenge: only 5% of nodes are fraudulent.
   Using `class_weight='balanced'` in Logistic Regression or focal loss in
   a GNN helps address this.

4. **Visualizing the fraud sub-graph** reveals the dense ring structure that
   distinguishes fraud from normal connectivity patterns.

5. In production, a full **GNN** (GCN, GraphSAGE, GAT) would replace
   hand-crafted features with learned message-passing layers, enabling
   end-to-end learning on raw transaction graphs.

In [ ]:
takeaways = [
    '1. Fraud accounts show higher clustering and distinct centrality vs legitimate ones.',
    '2. Hand-crafted graph features (degree, clustering, PageRank) capture fraud signal.',
    '3. GNNs learn equivalent features automatically via message passing.',
    '4. Class imbalance (5% fraud) requires balanced weighting or focal loss.',
    '5. Dense fraud rings are visible when visualising the fraud sub-graph.',
]
print('Key Takeaways')
print('=' * 60)
for t in takeaways:
    print(t)